# Finetuning and Adapters

The finetuning spectrum (most → least parameters changed):
1. Full finetuning — update all parameters
2. LoRA — low-rank adapter matrices (0.1-1% of params)
3. DreamBooth — subject-driven finetuning with prior preservation
4. Textual Inversion — optimize only a text embedding vector
5. Prompt engineering — zero parameter changes

Each has different compute/data requirements and use cases. This notebook covers the math and implementation of each.

**VRAM:** ~4-6 GB for demos. **Time:** ~2-3 hours.

**Key papers:** LoRA (Hu et al. 2022, 2106.09685), DreamBooth (Ruiz et al. 2023, 2208.12242).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math
from dataclasses import dataclass
from typing import Optional

## LoRA: Low-Rank Adaptation

**Math:** For a pretrained weight matrix W ∈ R^{d×k}, LoRA adds a low-rank decomposition:

$$W' = W + \Delta W = W + BA$$

where B ∈ R^{d×r}, A ∈ R^{r×k}, and r << min(d, k).

**Key properties:**
- Original W is frozen (no gradient computation = less memory)
- Only A and B are trained: r*(d+k) params vs d*k for full finetuning
- At inference, merge W' = W + BA — zero additional latency
- Initialization: A ~ N(0, σ²), B = 0 → ΔW = 0 at init (training starts from pretrained behavior)
- Scaling factor α/r controls the magnitude of the adaptation

**Why it works:** weight updates during finetuning have low intrinsic rank. Most of the information in ΔW can be captured by a rank-4 to rank-64 approximation.

In [ ]:
class LoRALinear(nn.Module):
    """Drop-in replacement for nn.Linear with LoRA adaptation."""
    def __init__(self, original: nn.Linear, rank: int = 4, alpha: float = 1.0) -> None:
        super().__init__()
        self.original = original
        self.original.weight.requires_grad_(False)
        if self.original.bias is not None:
            self.original.bias.requires_grad_(False)

        d_out, d_in = original.weight.shape
        self.lora_A = nn.Parameter(torch.randn(rank, d_in) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(d_out, rank))
        self.scaling = alpha / rank

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        base = self.original(x)
        lora = (x @ self.lora_A.T @ self.lora_B.T) * self.scaling
        return base + lora

    def merge(self) -> nn.Linear:
        """Merge LoRA weights into original for zero-overhead inference."""
        merged = nn.Linear(
            self.original.in_features, self.original.out_features,
            bias=self.original.bias is not None
        )
        merged.weight.data = self.original.weight.data + (self.lora_B @ self.lora_A) * self.scaling
        if self.original.bias is not None:
            merged.bias.data = self.original.bias.data
        return merged


def inject_lora(
    model: nn.Module,
    rank: int = 4,
    alpha: float = 1.0,
    target_modules: list[str] | None = None,
) -> nn.Module:
    """Replace target Linear layers with LoRA variants."""
    for name, module in list(model.named_modules()):
        if isinstance(module, nn.Linear):
            if target_modules is None or any(t in name for t in target_modules):
                parent_name = ".".join(name.split(".")[:-1])
                child_name = name.split(".")[-1]
                parent = model.get_submodule(parent_name) if parent_name else model
                setattr(parent, child_name, LoRALinear(module, rank=rank, alpha=alpha))
    return model


# --- Toy transformer-like model for demonstration ---

class ToyAttention(nn.Module):
    """Minimal multi-head self-attention with named Q/K/V projections."""
    def __init__(self, embed_dim: int = 256, num_heads: int = 4) -> None:
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.to_q = nn.Linear(embed_dim, embed_dim, bias=False)
        self.to_k = nn.Linear(embed_dim, embed_dim, bias=False)
        self.to_v = nn.Linear(embed_dim, embed_dim, bias=False)
        self.to_out = nn.Linear(embed_dim, embed_dim, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        q = self.to_q(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.to_k(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.to_v(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        scale = math.sqrt(self.head_dim)
        attn = (q @ k.transpose(-2, -1) / scale).softmax(dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(B, T, C)
        return self.to_out(out)


class ToyTransformerBlock(nn.Module):
    def __init__(self, embed_dim: int = 256, num_heads: int = 4) -> None:
        super().__init__()
        self.attn = ToyAttention(embed_dim, num_heads)
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.GELU(),
            nn.Linear(embed_dim * 4, embed_dim),
        )
        self.ln1 = nn.LayerNorm(embed_dim)
        self.ln2 = nn.LayerNorm(embed_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x


class ToyModel(nn.Module):
    """Two-layer transformer classifier for LoRA demonstration."""
    def __init__(self, vocab_size: int = 64, embed_dim: int = 256,
                 num_heads: int = 4, num_classes: int = 4) -> None:
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.blocks = nn.Sequential(
            ToyTransformerBlock(embed_dim, num_heads),
            ToyTransformerBlock(embed_dim, num_heads),
        )
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.embed(x)  # (B, T, C)
        h = self.blocks(h)
        return self.head(h.mean(dim=1))  # global average pool -> classify


def count_params(model: nn.Module) -> tuple[int, int]:
    """Return (total_params, trainable_params)."""
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


# Build model and inject LoRA
torch.manual_seed(42)
model = ToyModel()
total_before, trainable_before = count_params(model)
print(f"Before LoRA: {total_before:,} total, {trainable_before:,} trainable")

# Freeze everything first, then inject LoRA (LoRA unfreezes its own params)
for p in model.parameters():
    p.requires_grad_(False)

model = inject_lora(model, rank=4, alpha=4.0, target_modules=["to_q", "to_k", "to_v"])
total_after, trainable_after = count_params(model)
print(f"After LoRA:  {total_after:,} total, {trainable_after:,} trainable")
print(f"Reduction:   {trainable_after / total_before * 100:.2f}% of original params trained")


# --- Toy training loop to verify LoRA trains correctly ---
optimizer = torch.optim.Adam(
    [p for p in model.parameters() if p.requires_grad], lr=1e-3
)
losses = []
for step in range(50):
    x = torch.randint(0, 64, (8, 16))   # batch=8, seq_len=16
    y = torch.randint(0, 4, (8,))
    logits = model(x)
    loss = F.cross_entropy(logits, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

plt.figure(figsize=(7, 3))
plt.plot(losses)
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("LoRA training loss (toy task)")
plt.tight_layout()
plt.show()
print(f"Loss: {losses[0]:.4f} → {losses[-1]:.4f}")


# --- Merge and verify zero-overhead inference ---
def merge_all_lora(model: nn.Module) -> nn.Module:
    """Merge all LoRALinear layers back into plain nn.Linear."""
    for name, module in list(model.named_modules()):
        if isinstance(module, LoRALinear):
            parent_name = ".".join(name.split(".")[:-1])
            child_name = name.split(".")[-1]
            parent = model.get_submodule(parent_name) if parent_name else model
            setattr(parent, child_name, module.merge())
    return model


model.eval()
x_test = torch.randint(0, 64, (4, 16))
with torch.no_grad():
    out_pre_merge = model(x_test)

model_merged = merge_all_lora(model)
model_merged.eval()
with torch.no_grad():
    out_post_merge = model_merged(x_test)

max_diff = (out_pre_merge - out_post_merge).abs().max().item()
print(f"\nMax output diff pre/post merge: {max_diff:.2e}  (should be ~1e-6 or less)")

## LoRA with Diffusers

In practice, you use the `peft` library or diffusers' built-in LoRA support. Key concepts:
- **Target modules:** typically attention Q/K/V/O projections and sometimes feed-forward layers
- **Rank selection:** r=4 for style, r=16-64 for concept learning, r=128+ approaches full finetuning
- **Alpha:** typically set equal to rank (scaling = 1.0) or 2x rank
- **Training:** standard diffusion loss, but only LoRA params get gradients

In [ ]:
# Conceptual workflow (requires diffusers + peft installed)
# from diffusers import StableDiffusionPipeline
# from peft import LoraConfig, get_peft_model
#
# pipe = StableDiffusionPipeline.from_pretrained("runwayml/stable-diffusion-v1-5")
# lora_config = LoraConfig(
#     r=4, lora_alpha=4, target_modules=["to_q", "to_k", "to_v", "to_out.0"],
#     lora_dropout=0.0
# )
# unet = get_peft_model(pipe.unet, lora_config)
# unet.print_trainable_parameters()  # ~0.1% of total


# --- Working demo with our from-scratch LoRA ---

torch.manual_seed(0)
embed_dim = 128

# Small model, all frozen
small_model = nn.Sequential(
    nn.Linear(embed_dim, embed_dim),
    nn.GELU(),
    nn.Linear(embed_dim, embed_dim),
)
for p in small_model.parameters():
    p.requires_grad_(False)

# Inject LoRA into both linear layers
lora_model = inject_lora(small_model, rank=8, alpha=8.0)
optimizer = torch.optim.Adam(
    [p for p in lora_model.parameters() if p.requires_grad], lr=5e-3
)

# Synthetic task: map random inputs to a fixed target pattern
torch.manual_seed(1)
target_transform = torch.randn(embed_dim, embed_dim)

pre_losses, post_steps = [], 30
for step in range(post_steps):
    x = torch.randn(16, embed_dim)
    y = x @ target_transform
    out = lora_model(x)
    loss = F.mse_loss(out, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    pre_losses.append(loss.item())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss curve
axes[0].plot(pre_losses, color="steelblue")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("MSE Loss")
axes[0].set_title("LoRA adaptation loss")

# Visualize learned ΔW = BA for the first LoRA layer
lora_layer: LoRALinear = lora_model[0]  # type: ignore
delta_W = (lora_layer.lora_B @ lora_layer.lora_A).detach().numpy() * lora_layer.scaling
im = axes[1].imshow(delta_W[:32, :32], cmap="RdBu", aspect="auto")
plt.colorbar(im, ax=axes[1])
axes[1].set_title("Learned ΔW = BA (first 32×32 slice)")
axes[1].set_xlabel("d_in")
axes[1].set_ylabel("d_out")

plt.tight_layout()
plt.show()

# Rank of ΔW — confirm it is bounded by r
rank_estimate = np.linalg.matrix_rank(delta_W)
print(f"Rank of learned ΔW: {rank_estimate} (LoRA rank r=8, so rank ≤ 8 is expected)")

## DreamBooth: Subject-Driven Finetuning

DreamBooth teaches a model a specific subject (person, object, style) from 3-5 images.

**Key innovation:** prior preservation loss — while learning the subject, also generate and train on class-prior images to prevent language drift.

$$\mathcal{L} = \mathcal{L}_{\text{diffusion}}(\text{subject images}, \text{"[V] dog"}) + \lambda \cdot \mathcal{L}_{\text{diffusion}}(\text{generated class images}, \text{"dog"})$$

Where `[V]` is a rare token identifier bound to the subject.

In [ ]:
@dataclass
class DreamBoothConfig:
    instance_prompt: str = "a photo of sks dog"
    class_prompt: str = "a photo of dog"
    prior_loss_weight: float = 1.0
    num_class_images: int = 200


def add_noise(
    x: torch.Tensor,
    t: torch.Tensor,
    noise: torch.Tensor,
    beta_start: float = 1e-4,
    beta_end: float = 0.02,
    T: int = 1000,
) -> torch.Tensor:
    """Forward diffusion: add noise at timestep t using a linear schedule."""
    betas = torch.linspace(beta_start, beta_end, T)
    alphas = 1.0 - betas
    alphas_cumprod = torch.cumprod(alphas, dim=0)
    # Gather sqrt_alpha_cumprod and sqrt_one_minus for each sample
    a_bar = alphas_cumprod[t]  # (B,)
    # Reshape for broadcasting: (B, 1, 1, ...) to match x shape
    view_shape = (-1,) + (1,) * (x.ndim - 1)
    sqrt_a = a_bar.sqrt().view(view_shape)
    sqrt_one_minus_a = (1.0 - a_bar).sqrt().view(view_shape)
    return sqrt_a * x + sqrt_one_minus_a * noise


class ToyDiffusionModel(nn.Module):
    """Minimal noise predictor for DreamBooth demonstration."""
    def __init__(self, img_channels: int = 4, T: int = 1000) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(img_channels * 8 * 8 + 1, 256),
            nn.GELU(),
            nn.Linear(256, img_channels * 8 * 8),
        )
        self.img_channels = img_channels

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        B = x.shape[0]
        t_norm = t.float().unsqueeze(1) / 1000.0  # (B, 1)
        x_flat = x.view(B, -1)                    # (B, C*H*W)
        inp = torch.cat([x_flat, t_norm], dim=1)
        out = self.net(inp)
        return out.view_as(x)


def dreambooth_training_step(
    model: nn.Module,
    instance_batch: torch.Tensor,
    class_batch: torch.Tensor,
    config: DreamBoothConfig,
) -> torch.Tensor:
    """Single DreamBooth training step with prior preservation."""
    # Instance loss (learn the subject)
    t = torch.randint(0, 1000, (instance_batch.shape[0],))
    noise = torch.randn_like(instance_batch)
    noisy = add_noise(instance_batch, t, noise)
    pred_instance = model(noisy, t)
    loss_instance = F.mse_loss(pred_instance, noise)

    # Prior preservation loss (prevent forgetting)
    t_class = torch.randint(0, 1000, (class_batch.shape[0],))
    noise_class = torch.randn_like(class_batch)
    noisy_class = add_noise(class_batch, t_class, noise_class)
    pred_class = model(noisy_class, t_class)
    loss_prior = F.mse_loss(pred_class, noise_class)

    return loss_instance + config.prior_loss_weight * loss_prior


# --- Demonstrate dual-loss training ---
torch.manual_seed(7)
cfg = DreamBoothConfig(prior_loss_weight=1.0)
diff_model = ToyDiffusionModel(img_channels=4)
opt = torch.optim.Adam(diff_model.parameters(), lr=1e-3)

H = W = 8
instance_losses, prior_losses_log, total_losses = [], [], []

for step in range(40):
    # Toy 'instance' images (subject) and 'class' images (prior)
    instance_batch = torch.randn(4, 4, H, W) * 0.8 + 0.5   # slightly shifted
    class_batch = torch.randn(4, 4, H, W)                   # standard

    # Log individual loss components
    t_i = torch.randint(0, 1000, (4,))
    n_i = torch.randn_like(instance_batch)
    with torch.no_grad():
        li = F.mse_loss(diff_model(add_noise(instance_batch, t_i, n_i), t_i), n_i).item()
        t_c = torch.randint(0, 1000, (4,))
        n_c = torch.randn_like(class_batch)
        lp = F.mse_loss(diff_model(add_noise(class_batch, t_c, n_c), t_c), n_c).item()

    total = dreambooth_training_step(diff_model, instance_batch, class_batch, cfg)
    opt.zero_grad()
    total.backward()
    opt.step()

    instance_losses.append(li)
    prior_losses_log.append(lp)
    total_losses.append(total.item())


# --- Plot: dual loss components ---
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(instance_losses, label="Instance loss", color="crimson")
axes[0].plot(prior_losses_log, label="Prior preservation loss", color="steelblue")
axes[0].plot(total_losses, label="Total loss", color="black", linestyle="--")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("MSE")
axes[0].set_title("DreamBooth: dual loss components")
axes[0].legend()

# Conceptual diagram: prior preservation prevents forgetting
axes[1].axis("off")
diagram_text = (
    "Prior Preservation Concept\n\n"
    "Without prior loss:\n"
    "  Model learns [V] dog so hard\n"
    "  it forgets what 'dog' means\n"
    "  → language drift\n\n"
    "With prior loss (λ·L_prior):\n"
    "  Simultaneously trained on\n"
    "  generated 'dog' images\n"
    "  → preserves prior knowledge\n\n"
    "Key: λ=1.0 balances instance\n"
    "vs class-prior signal"
)
axes[1].text(0.05, 0.95, diagram_text, transform=axes[1].transAxes,
             fontsize=11, verticalalignment="top", fontfamily="monospace",
             bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.8))

plt.tight_layout()
plt.show()
print(f"Instance loss: {instance_losses[0]:.4f} → {instance_losses[-1]:.4f}")
print(f"Prior loss:    {prior_losses_log[0]:.4f} → {prior_losses_log[-1]:.4f}")

## Textual Inversion

Lightest-weight adaptation: freeze the entire model, optimize only a single text embedding vector.

- Add a new token `<my-concept>` to the vocabulary
- Initialize its embedding (random or from a similar word)
- Optimize only this embedding using the standard diffusion loss
- Result: the new token "points to" the concept in the model's latent space

**Trade-off:** very cheap to train, but limited expressiveness — works for styles and simple concepts, struggles with complex subjects.

In [ ]:
from sklearn.decomposition import PCA  # type: ignore


class TextualInversionEmbeddings(nn.Module):
    """Simulated CLIP-like text embedding table with one learnable concept token."""
    def __init__(self, vocab_size: int = 50, embed_dim: int = 64) -> None:
        super().__init__()
        # Pre-trained embeddings (frozen)
        self.vocab_embed = nn.Embedding(vocab_size, embed_dim)
        nn.init.normal_(self.vocab_embed.weight, std=0.02)
        self.vocab_embed.weight.requires_grad_(False)

        # Single learnable concept embedding
        self.concept_embed = nn.Parameter(torch.randn(1, embed_dim) * 0.01)

    def get_concept(self) -> torch.Tensor:
        return self.concept_embed.squeeze(0)  # (embed_dim,)

    def get_all_vocab(self) -> torch.Tensor:
        return self.vocab_embed.weight.data  # (vocab_size, embed_dim)


def ti_training_step(
    embeddings: TextualInversionEmbeddings,
    target_style: torch.Tensor,  # synthetic 'target' the new token should represent
) -> torch.Tensor:
    """Optimize concept embedding to minimize distance to target style representation."""
    concept = embeddings.get_concept()
    # Simplified loss: cosine similarity to a synthetic style centroid
    loss = 1.0 - F.cosine_similarity(concept.unsqueeze(0), target_style.unsqueeze(0))
    return loss.mean()


# --- Training loop ---
torch.manual_seed(42)
embed_dim = 64
ti_model = TextualInversionEmbeddings(vocab_size=50, embed_dim=embed_dim)
# Confirm only concept_embed is trainable
trainable = sum(p.numel() for p in ti_model.parameters() if p.requires_grad)
total_ti = sum(p.numel() for p in ti_model.parameters())
print(f"TI trainable params: {trainable} / {total_ti} ({trainable/total_ti*100:.2f}%)")

# Synthetic target style: a specific direction in embedding space
torch.manual_seed(99)
target_style = F.normalize(torch.randn(embed_dim), dim=0)

ti_optimizer = torch.optim.Adam([ti_model.concept_embed], lr=0.05)
embedding_trajectory: list[np.ndarray] = []

for step in range(80):
    loss = ti_training_step(ti_model, target_style)
    ti_optimizer.zero_grad()
    loss.backward()
    ti_optimizer.step()
    if step % 5 == 0:
        embedding_trajectory.append(ti_model.get_concept().detach().numpy().copy())


# --- Visualize embedding trajectory (PCA) ---
vocab_embeddings = ti_model.get_all_vocab().numpy()  # (50, 64)
traj_arr = np.stack(embedding_trajectory)            # (num_checkpoints, 64)

# Fit PCA on vocab + trajectory points
all_points = np.vstack([vocab_embeddings, traj_arr, target_style.numpy()[None, :]])
pca = PCA(n_components=2)
all_2d = pca.fit_transform(all_points)

vocab_2d = all_2d[:50]
traj_2d = all_2d[50:50 + len(traj_arr)]
target_2d = all_2d[-1]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# PCA trajectory plot
axes[0].scatter(vocab_2d[:, 0], vocab_2d[:, 1], c="lightgray", s=40, label="Vocab tokens", zorder=1)
axes[0].scatter(*target_2d, c="gold", s=200, marker="*", label="Target style", zorder=3, edgecolors="black")
colors = plt.cm.cool(np.linspace(0, 1, len(traj_2d)))  # type: ignore
for i, (pt, col) in enumerate(zip(traj_2d, colors)):
    axes[0].scatter(*pt, c=[col], s=60, zorder=2)
    if i > 0:
        axes[0].annotate("", xy=traj_2d[i], xytext=traj_2d[i - 1],
                         arrowprops=dict(arrowstyle="->", color="steelblue", lw=1.2))
axes[0].scatter(*traj_2d[0], c="blue", s=100, marker="o", label="Init", zorder=4, edgecolors="black")
axes[0].scatter(*traj_2d[-1], c="red", s=100, marker="o", label="Final", zorder=4, edgecolors="black")
axes[0].set_title("Textual Inversion: embedding trajectory (PCA)")
axes[0].legend(fontsize=8)
axes[0].set_xlabel("PC 1")
axes[0].set_ylabel("PC 2")

# Nearest-neighbor similarity before/after
init_embed = torch.tensor(embedding_trajectory[0])
final_embed = torch.tensor(embedding_trajectory[-1])
vocab_t = ti_model.get_all_vocab()

def top_k_similar(query: torch.Tensor, vocab: torch.Tensor, k: int = 5) -> list[tuple[int, float]]:
    """Return top-k most similar vocab indices and cosine similarities."""
    sims = F.cosine_similarity(query.unsqueeze(0), vocab).detach().tolist()
    ranked = sorted(enumerate(sims), key=lambda x: x[1], reverse=True)
    return ranked[:k]

init_sims = [s for _, s in top_k_similar(init_embed, vocab_t)]
final_sims = [s for _, s in top_k_similar(final_embed, vocab_t)]
target_sim_init = F.cosine_similarity(init_embed.unsqueeze(0), target_style.unsqueeze(0)).item()
target_sim_final = F.cosine_similarity(final_embed.unsqueeze(0), target_style.unsqueeze(0)).item()

axes[1].bar(["Init sim\nto target", "Final sim\nto target"],
            [target_sim_init, target_sim_final], color=["steelblue", "crimson"])
axes[1].set_ylim(-0.1, 1.05)
axes[1].set_ylabel("Cosine similarity")
axes[1].set_title("Concept embedding similarity to target style")
for i, v in enumerate([target_sim_init, target_sim_final]):
    axes[1].text(i, v + 0.02, f"{v:.3f}", ha="center", fontsize=11)

plt.tight_layout()
plt.show()
print(f"Cosine sim to target: {target_sim_init:.4f} (init) → {target_sim_final:.4f} (final)")

## When to Finetune vs When to Fix Data

Decision tree for diagnosing model output problems:
1. **Model outputs wrong content** → check training data distribution, likely a data mixing issue
2. **Model outputs right content, wrong style** → LoRA on style-specific subset (r=4-8)
3. **Model doesn't know a specific subject** → DreamBooth with 3-5 images
4. **Model knows the concept but uses wrong word** → Textual Inversion
5. **Model quality is generally low** → more/better data, not finetuning

**Rule of thumb:** fix data first, finetune second. Finetuning on bad data makes things worse.

In [ ]:
def diagnose_model_issue(
    outputs: list[str],
    expected: list[str],
    training_data_size: int,
    has_concept_in_data: bool,
) -> str:
    """Suggest the most appropriate intervention based on symptom heuristics."""
    # Simple keyword heuristics — in practice this would be a human judgment call
    style_mismatch = any(
        any(kw in o.lower() for kw in ["style", "color", "tone", "texture"])
        for o in outputs
    )
    subject_unknown = not has_concept_in_data
    low_data = training_data_size < 1000
    semantic_mismatch = any(
        any(kw in o.lower() for kw in ["wrong", "different", "unrelated"])
        for o in outputs
    )

    if semantic_mismatch and low_data:
        return "FIX DATA: outputs semantically wrong and data is small — collect more/better data first."
    if semantic_mismatch and not low_data:
        return "CHECK DATA DISTRIBUTION: likely a data mixing/labeling issue. Fix before finetuning."
    if subject_unknown:
        return "DREAMBOOTH: concept not in training data — use DreamBooth with 3-5 subject images."
    if style_mismatch:
        return "LoRA (r=4-8): style mismatch — lightweight LoRA on style-specific subset."
    return "PROMPT ENGINEERING: issue may be prompt-solvable — try before any finetuning."


@dataclass
class FinetuningCostEstimate:
    method: str
    params_trained: int
    params_trained_pct: float
    estimated_vram_gb: float
    estimated_hours: float


def estimate_finetuning_cost(
    method: str,
    model_size_b: float,   # model parameters in billions
    dataset_size: int,     # number of training samples
    gpu_type: str = "A100",
) -> FinetuningCostEstimate:
    """Rough cost estimate for different finetuning methods."""
    total_params = int(model_size_b * 1e9)
    # Throughput in samples/sec by GPU tier
    gpu_throughput: dict[str, float] = {"A100": 80.0, "V100": 30.0, "RTX3090": 20.0, "RTX2080": 10.0}
    tput = gpu_throughput.get(gpu_type, 20.0)

    method_cfg: dict[str, dict] = {
        "full": {"param_frac": 1.0, "vram_mult": 16.0, "speed_mult": 1.0},
        "lora_r4": {"param_frac": 0.001, "vram_mult": 3.0, "speed_mult": 4.0},
        "lora_r64": {"param_frac": 0.01, "vram_mult": 4.0, "speed_mult": 3.0},
        "dreambooth": {"param_frac": 1.0, "vram_mult": 14.0, "speed_mult": 0.5},
        "textual_inversion": {"param_frac": 1e-6, "vram_mult": 2.0, "speed_mult": 8.0},
    }
    cfg = method_cfg.get(method, method_cfg["lora_r4"])
    base_vram = model_size_b * 2  # fp16 model weight bytes (approx GB)
    vram_gb = base_vram * cfg["vram_mult"]
    effective_tput = tput * cfg["speed_mult"]
    hours = dataset_size / effective_tput / 3600.0
    params_trained = int(total_params * cfg["param_frac"])

    return FinetuningCostEstimate(
        method=method,
        params_trained=params_trained,
        params_trained_pct=cfg["param_frac"] * 100,
        estimated_vram_gb=vram_gb,
        estimated_hours=hours,
    )


# --- Diagnosis examples ---
print("=== Diagnosis Examples ===")
print(diagnose_model_issue(
    outputs=["wrong colors, style is off"],
    expected=["vibrant oil-paint style"],
    training_data_size=5000,
    has_concept_in_data=True,
))
print(diagnose_model_issue(
    outputs=["completely unrelated content"],
    expected=["portraits"],
    training_data_size=200,
    has_concept_in_data=False,
))
print(diagnose_model_issue(
    outputs=["good quality output"],
    expected=["slightly different composition"],
    training_data_size=10000,
    has_concept_in_data=True,
))

print()

# --- Comparison table ---
methods = ["full", "lora_r4", "lora_r64", "dreambooth", "textual_inversion"]
use_cases = [
    "Domain shift, task adaptation",
    "Style, texture (3-5 images ok)",
    "Subject/concept learning",
    "Specific person/object (3-5 imgs)",
    "New token for a style/concept",
]
data_needs = ["10k+", "50-500", "200-2k", "3-5", "3-20"]

print(f"{'Method':<22} {'% Params':<12} {'VRAM (GB)':<12} {'Hours (1M samples)':<22} {'Data needed':<12} {'Use case'}")
print("-" * 110)
for method, use_case, data in zip(methods, use_cases, data_needs):
    est = estimate_finetuning_cost(method, model_size_b=0.9, dataset_size=1_000_000, gpu_type="A100")
    print(f"{method:<22} {est.params_trained_pct:<12.4f} {est.estimated_vram_gb:<12.1f} {est.estimated_hours:<22.1f} {data:<12} {use_case}")

## Checkpoint

Key takeaways:
- **LoRA math**: W' = W + BA, rank r << min(d_in, d_out), merge for zero-overhead inference
- **LoRA in practice**: r=4 for style, r=16-64 for concepts, target attention projections first
- **DreamBooth**: prior preservation loss prevents catastrophic forgetting, needs only 3-5 images
- **Textual Inversion**: cheapest adaptation, optimize one embedding vector, limited expressiveness
- **Decision order**: fix data → prompt engineering → textual inversion → LoRA → DreamBooth → full finetuning
- **Adapter composition**: multiple LoRA adapters can be merged (weighted sum of ΔW matrices) for multi-concept generation

Further reading: LoRA (2106.09685), DreamBooth (2208.12242)